# RiceLeafs ResNet50V2 — v8 Final Robust

Notebook final untuk klasifikasi **BrownSpot, Healthy, Hispa, dan LeafBlast** dengan backbone **ResNet50V2**.

Perbaikan v8 dibanding v7:

- split 80/10/10 tetap *group-aware* dan memakai versi/seed baru;
- audit kandidat gambar mirip lintas kelas tanpa menghapus atau mengubah label otomatis;
- tidak menumpuk focal loss, oversampling, dan class weight sekaligus;
- memakai categorical cross-entropy dengan label smoothing dan class weight *effective number* yang dibatasi;
- checkpoint dipilih dari macro-F1, geometric-mean F1, minimum F1, dan lower confidence bound recall;
- checkpoint dan riwayat aman untuk dilanjutkan setelah runtime terputus;
- TTA dipilih dengan aturan toleransi agar pemerataan kelas tidak kalah karena selisih skor sangat kecil;
- test set hanya dievaluasi setelah model, TTA, temperature, dan threshold selesai dipilih dari validation.

> Jalankan menggunakan GPU Colab melalui **Runtime → Restart session**, kemudian **Runtime → Run all**. Notebook ini tidak menjanjikan angka tertentu; hasil akhir tetap ditentukan oleh data holdout.


In [ ]:
# instal library
%pip install -q -U kagglehub ImageHash

## 2. Import

In [ ]:
# import library
from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import random
import shutil
import zipfile

import imagehash
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive, files
from PIL import Image, UnidentifiedImageError
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

## 3. Konfigurasi

In [ ]:
# konfigurasi final v8: satu eksperimen fresh, reproducible, dan tidak memakai test untuk selection
SEED = 20260803
SPLIT_SEED = 20260803
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 8

# Batas maksimum. Persistent early stopping selalu mengembalikan checkpoint terbaik.
HEAD_EPOCHS = 15
FINE_TUNE_EPOCHS = 40
FINE_TUNE_LAYERS = 120
HEAD_LR = 2e-4
FINE_TUNE_LR = 3e-6

WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.30
DENSE_UNITS = 512

# Satu mekanisme koreksi imbalance saja: effective-number class weight.
# Tidak ada oversampling agar contoh Hispa yang sama tidak diulang berlebihan.
CLASS_OVERSAMPLE_FACTORS = {
    'BrownSpot': 1.00,
    'Healthy': 1.00,
    'Hispa': 1.00,
    'LeafBlast': 1.00,
}
HISPA_OVERSAMPLE_FACTOR = 1.00  # kompatibilitas metadata lama
CLASS_WEIGHT_POWER = 0.00       # kompatibilitas metadata lama
FOCAL_GAMMA = 0.00             # v8 tidak menggunakan focal loss
LABEL_SMOOTHING = 0.04
EFFECTIVE_NUMBER_BETA = 0.999
MIN_CLASS_WEIGHT = 0.70
MAX_CLASS_WEIGHT = 1.45

# Objective checkpoint: macro sebagai utama, geomean untuk pemerataan,
# serta lower confidence bound recall agar satu kelas kecil tidak mudah terlihat "bagus" karena noise.
SELECTION_MACRO_WEIGHT = 0.45
SELECTION_GEOMEAN_WEIGHT = 0.30
SELECTION_WORST_CLASS_WEIGHT = 0.10
SELECTION_RECALL_LCB_WEIGHT = 0.15
SELECTION_MIN_RECALL_WEIGHT = SELECTION_RECALL_LCB_WEIGHT
SELECTION_HISPA_WEIGHT = 0.00
WILSON_Z = 1.2815515655446004  # one-sided 90% lower confidence bound
EARLY_STOPPING_PATIENCE = 10
LR_PATIENCE = 4
SELECTION_MIN_DELTA = 0.001

PHASH_DISTANCE = 3
RUN_NEAR_DUPLICATE_GROUPING = True

LOW_CONF_THRESHOLD = 0.50
TARGET_ACCEPTED_ACCURACY = 0.85
MIN_ACCEPTED_COVERAGE = 0.40
MIN_ACCEPTED_COUNT = 50
MIN_CONFIDENCE_MARGIN = 0.10
AUTO_SELECT_TTA = True
USE_TTA = True
TTA_SELECTION_TOLERANCE = 0.005

# Jangan ganti FORCE_RETRAIN menjadi True untuk sekadar melanjutkan runtime.
# PROJECT_NAME baru memastikan tidak membaca checkpoint v6/v7.
FORCE_RETRAIN = False
DATASET_HANDLE = 'shayanriyaz/riceleafs'
PROJECT_NAME = 'RiceLeafs_ResNet50V2_v8_FinalRobust'

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Image size:', IMAGE_SIZE, '| Batch:', BATCH_SIZE)
print('Project version:', PROJECT_NAME)
print('Loss: categorical cross-entropy | label smoothing:', LABEL_SMOOTHING)
print('Split seed:', SPLIT_SEED, '| Fine-tune layers:', FINE_TUNE_LAYERS)


## 4. Google Drive

In [ ]:
# mount drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive') / PROJECT_NAME
MODEL_DIR = DRIVE_ROOT / 'model_bundle'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RESULT_DIR = DRIVE_ROOT / 'evaluation'
AUDIT_DIR = DRIVE_ROOT / 'dataset_audit'

for folder in [MODEL_DIR, CHECKPOINT_DIR, RESULT_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if FORCE_RETRAIN:
    for folder in [MODEL_DIR, CHECKPOINT_DIR, RESULT_DIR]:
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)

print('Project:', DRIVE_ROOT)

## 5. Unduh Dataset

In [ ]:
# unduh data dari kaggle
download_path = Path(kagglehub.dataset_download(DATASET_HANDLE)).resolve()

possible_roots = [download_path] + [p for p in download_path.rglob('*') if p.is_dir()]
candidates = [
    p for p in possible_roots
    if (p / 'train').is_dir() and (p / 'validation').is_dir()
]
if not candidates:
    raise FileNotFoundError('Folder train dan validation tidak ditemukan.')

DATA_ROOT = sorted(candidates, key=lambda p: len(p.parts))[0]
VISIBLE_ROOT = Path('/content/RiceLeafs')
if VISIBLE_ROOT.is_symlink():
    VISIBLE_ROOT.unlink()
elif VISIBLE_ROOT.exists():
    shutil.rmtree(VISIBLE_ROOT)
VISIBLE_ROOT.symlink_to(DATA_ROOT, target_is_directory=True)

print('Dataset:', DATA_ROOT)
print('Shortcut:', VISIBLE_ROOT)

## 6. Inventaris Dataset

In [ ]:
# inventaris data
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
source_dirs = {
    'train_original': DATA_ROOT / 'train',
    'validation_original': DATA_ROOT / 'validation',
}
class_names = sorted([p.name for p in source_dirs['train_original'].iterdir() if p.is_dir()])
if len(class_names) != 4:
    raise ValueError(f'Jumlah kelas tidak sesuai: {class_names}')

rows = []
for source_split, split_dir in source_dirs.items():
    for class_name in class_names:
        for image_path in sorted((split_dir / class_name).rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({
                    'path': str(image_path),
                    'relative_path': str(image_path.relative_to(DATA_ROOT)),
                    'source_split': source_split,
                    'class_name': class_name,
                    'size_bytes': image_path.stat().st_size,
                })

data_df = pd.DataFrame(rows)
print('Kelas:', class_names)
print('Jumlah gambar:', len(data_df))
display(data_df.groupby(['source_split', 'class_name']).size().reset_index(name='count'))
if len(data_df) != 3355:
    raise ValueError(f'Jumlah gambar bukan 3.355: {len(data_df)}')

## 7. Audit File dan Hash

In [ ]:
# audit gambar
def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, 'rb') as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def inspect_image(file_path):
    with Image.open(file_path) as image:
        image.load()
        rgb = image.convert('RGB')
        phash = imagehash.phash(rgb, hash_size=8)
        return {
            'width': image.width,
            'height': image.height,
            'mode': image.mode,
            'sha256': sha256_file(file_path),
            'phash': str(phash),
            'phash_int': int(str(phash), 16),
        }

valid_rows, corrupt_rows = [], []
for number, row in enumerate(data_df.itertuples(index=False), start=1):
    try:
        valid_rows.append({**row._asdict(), **inspect_image(row.path)})
    except Exception as error:
        corrupt_rows.append({'path': row.path, 'error': str(error)})
    if number % 250 == 0:
        print(f'Audit {number:,}/{len(data_df):,}')

audit_df = pd.DataFrame(valid_rows)
corrupt_df = pd.DataFrame(corrupt_rows, columns=['path', 'error'])
audit_df.to_csv(AUDIT_DIR / 'dataset_audit.csv', index=False)
corrupt_df.to_csv(AUDIT_DIR / 'corrupt_images.csv', index=False)

print('Gambar valid:', len(audit_df))
print('Gambar rusak:', len(corrupt_df))
print('Resolusi:', audit_df['width'].min(), '-', audit_df['width'].max(), 'x', audit_df['height'].min(), '-', audit_df['height'].max())
if len(corrupt_df):
    display(corrupt_df.head(20))

## 8. Bersihkan Duplikasi Persis

In [ ]:
# cek label konflik
label_conflicts = audit_df.groupby('sha256')['class_name'].nunique()
conflicting_hashes = label_conflicts[label_conflicts > 1].index.tolist()
if conflicting_hashes:
    conflict_df = audit_df[audit_df['sha256'].isin(conflicting_hashes)].sort_values('sha256')
    conflict_df.to_csv(AUDIT_DIR / 'label_conflicts.csv', index=False)
    display(conflict_df)
    raise ValueError('Ada gambar identik dengan label berbeda.')

exact_duplicate_mask = audit_df.duplicated(subset=['sha256'], keep='first')
exact_duplicates_df = audit_df[exact_duplicate_mask].copy()
exact_duplicates_df.to_csv(AUDIT_DIR / 'exact_duplicates_removed.csv', index=False)
clean_df = audit_df[~exact_duplicate_mask].copy().reset_index(drop=True)

print('Duplikasi persis:', len(exact_duplicates_df))
print('Data unik:', len(clean_df))

## 9. Audit Near-Duplicate dan Kelompok Gambar Serupa

Kandidat near-duplicate lintas kelas hanya dilaporkan untuk audit. Notebook **tidak menggabungkan, menghapus, atau mengganti labelnya secara otomatis** karena kemiripan background dapat menghasilkan pHash yang dekat walaupun penyakit berbeda.


In [ ]:
# union-find + BK-tree untuk grouping dan audit near-duplicate
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, first, second):
        first, second = self.find(first), self.find(second)
        if first == second:
            return
        if self.rank[first] < self.rank[second]:
            first, second = second, first
        self.parent[second] = first
        if self.rank[first] == self.rank[second]:
            self.rank[first] += 1


class BKNode:
    def __init__(self, value, index):
        self.value, self.index, self.children = value, index, {}


class BKTree:
    def __init__(self):
        self.root = None

    @staticmethod
    def distance(first, second):
        return (first ^ second).bit_count()

    def add(self, value, index):
        if self.root is None:
            self.root = BKNode(value, index)
            return
        node = self.root
        while True:
            distance = self.distance(value, node.value)
            if distance not in node.children:
                node.children[distance] = BKNode(value, index)
                return
            node = node.children[distance]

    def query(self, value, radius):
        results = []

        def search(node):
            if node is None:
                return
            distance = self.distance(value, node.value)
            if distance <= radius:
                results.append((node.index, distance))
            lower, upper = distance - radius, distance + radius
            for edge, child in node.children.items():
                if lower <= edge <= upper:
                    search(child)

        search(self.root)
        return results


# Audit lintas kelas; tidak dipakai sebagai aturan penghapusan otomatis.
cross_class_candidates = []
global_tree = BKTree()
for dataframe_index, row in clean_df.iterrows():
    hash_value = int(row['phash_int'])
    for matched_index, distance in global_tree.query(hash_value, PHASH_DISTANCE):
        matched = clean_df.loc[matched_index]
        if matched['class_name'] != row['class_name']:
            cross_class_candidates.append({
                'index_a': int(matched_index),
                'index_b': int(dataframe_index),
                'class_a': matched['class_name'],
                'class_b': row['class_name'],
                'path_a': matched['relative_path'],
                'path_b': row['relative_path'],
                'phash_distance': int(distance),
            })
    global_tree.add(hash_value, dataframe_index)

cross_class_near_duplicates_df = pd.DataFrame(cross_class_candidates)
cross_class_near_duplicates_df.to_csv(
    AUDIT_DIR / 'cross_class_near_duplicate_candidates.csv',
    index=False,
)
print('Kandidat near-duplicate lintas kelas:', len(cross_class_near_duplicates_df))
if len(cross_class_near_duplicates_df):
    display(cross_class_near_duplicates_df.head(30))

# Hanya near-duplicate dengan label sama yang dikelompokkan agar tidak bocor antar split.
union_find = UnionFind(len(clean_df))
if RUN_NEAR_DUPLICATE_GROUPING:
    for class_name in class_names:
        indices = clean_df.index[clean_df['class_name'] == class_name].tolist()
        tree = BKTree()
        for dataframe_index in indices:
            hash_value = int(clean_df.at[dataframe_index, 'phash_int'])
            for matched_index, _ in tree.query(hash_value, PHASH_DISTANCE):
                union_find.union(dataframe_index, matched_index)
            tree.add(hash_value, dataframe_index)
        print(class_name, len(indices))

clean_df['near_duplicate_group'] = [
    f"{row.class_name}_{union_find.find(index)}"
    for index, row in clean_df.iterrows()
]
group_sizes = clean_df.groupby('near_duplicate_group').size().sort_values(ascending=False)
print('Jumlah grup:', len(group_sizes))
print('Grup terbesar:', int(group_sizes.max()))
display(group_sizes.head(20).rename('size').reset_index())
clean_df.to_csv(AUDIT_DIR / 'clean_dataset_with_groups.csv', index=False)


## 10. Split Group-Aware 80/10/10

Split v8 memakai seed baru dan disimpan sebagai manifest. Test tidak dipakai pada training, checkpoint selection, TTA selection, temperature scaling, atau threshold selection.


In [ ]:
# pilih fold dengan distribusi paling dekat ke keseluruhan dataset
def choose_best_fold(dataframe, n_splits, seed):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    labels = dataframe['class_name'].to_numpy()
    groups = dataframe['near_duplicate_group'].to_numpy()
    overall = dataframe['class_name'].value_counts(normalize=True).reindex(class_names, fill_value=0).to_numpy()
    candidates = []
    for fold, (train_indices, holdout_indices) in enumerate(splitter.split(dataframe, labels, groups)):
        holdout = dataframe.iloc[holdout_indices]
        distribution = holdout['class_name'].value_counts(normalize=True).reindex(class_names, fill_value=0).to_numpy()
        distribution_gap = float(np.abs(distribution - overall).sum())
        size_gap = abs(len(holdout) / len(dataframe) - 1 / n_splits)
        candidates.append((distribution_gap + size_gap, fold, train_indices, holdout_indices))
    return min(candidates, key=lambda item: item[0])


_, test_fold, train_val_indices, test_indices = choose_best_fold(clean_df, 10, SPLIT_SEED)
train_val_df = clean_df.iloc[train_val_indices].copy()
test_df = clean_df.iloc[test_indices].copy()

_, val_fold, train_indices, val_indices = choose_best_fold(train_val_df, 9, SPLIT_SEED + 1)
train_df = train_val_df.iloc[train_indices].copy()
val_df = train_val_df.iloc[val_indices].copy()

train_df['split'] = 'train'
val_df['split'] = 'validation'
test_df['split'] = 'test'
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

if split_df.groupby('near_duplicate_group')['split'].nunique().max() != 1:
    raise ValueError('Grup gambar serupa tersebar ke beberapa split.')
for split_name in ['train', 'validation', 'test']:
    observed = set(split_df.loc[split_df['split'] == split_name, 'class_name'])
    if observed != set(class_names):
        raise ValueError(f'Kelas tidak lengkap pada {split_name}.')

manifest_columns = [
    'relative_path', 'sha256', 'class_name', 'near_duplicate_group', 'split'
]
SPLIT_MANIFEST = AUDIT_DIR / 'final_split_manifest_v8.csv'
split_df[manifest_columns].sort_values(['split', 'class_name', 'relative_path']).to_csv(
    SPLIT_MANIFEST,
    index=False,
)

split_fingerprint = hashlib.sha256(
    split_df[manifest_columns]
    .sort_values(['relative_path'])
    .to_csv(index=False)
    .encode('utf-8')
).hexdigest()

display(split_df.groupby(['split', 'class_name']).size().reset_index(name='count'))
display(split_df['split'].value_counts().rename_axis('split').reset_index(name='count'))
print('Test fold:', test_fold, '| Validation fold:', val_fold)
print('Split fingerprint:', split_fingerprint)
print('Manifest:', SPLIT_MANIFEST)


## 11. Contoh Gambar

In [ ]:
# contoh gambar
sample_rows = split_df.groupby('class_name').sample(n=2, random_state=SEED).reset_index(drop=True)
plt.figure(figsize=(12, 8))
for index, row in sample_rows.iterrows():
    with Image.open(row['path']) as image:
        image = image.convert('RGB')
    plt.subplot(len(class_names), 2, index + 1)
    plt.imshow(image)
    plt.title(f"{row['class_name']} | {row['split']}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 12. Pipeline TensorFlow

In [ ]:
# pipeline data: label one-hot agar loss built-in dapat memakai label smoothing
class_to_index = {name: index for index, name in enumerate(class_names)}
AUTOTUNE = tf.data.AUTOTUNE


def decode_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(
        image,
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        antialias=True,
    )
    image = tf.cast(image, tf.float32)
    label = tf.one_hot(tf.cast(label, tf.int32), depth=len(class_names))
    return image, label


def dataframe_to_dataset(dataframe, training=False):
    paths = dataframe['path'].astype(str).tolist()
    labels = dataframe['class_name'].map(class_to_index).astype(int).tolist()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(
            len(paths), seed=SEED, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    options = tf.data.Options()
    options.experimental_deterministic = not training
    dataset = dataset.with_options(options)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# Tidak ada oversampling pada v8. Nama train_model_df dipertahankan agar ekspor kompatibel.
train_model_df = train_df.copy().reset_index(drop=True)
train_ds = dataframe_to_dataset(train_model_df, training=True)
val_ds = dataframe_to_dataset(val_df)
test_ds = dataframe_to_dataset(test_df)

distribution_table = pd.concat(
    [
        train_df['class_name'].value_counts().rename('train_original'),
        train_model_df['class_name'].value_counts().rename('train_model'),
    ],
    axis=1,
).fillna(0).astype(int).reindex(class_names)
distribution_table['multiplier'] = (
    distribution_table['train_model'] / distribution_table['train_original']
)
display(distribution_table)
print('Jumlah train:', len(train_df), '->', len(train_model_df))
print(train_ds.element_spec)


## 13. Class Weight

In [ ]:
# Effective-number class weight: satu-satunya koreksi imbalance pada v8.
train_numeric_labels = train_df['class_name'].map(class_to_index).astype(int).to_numpy()
classes = np.arange(len(class_names))
class_counts = np.bincount(train_numeric_labels, minlength=len(class_names)).astype(np.float64)

effective_numbers = 1.0 - np.power(EFFECTIVE_NUMBER_BETA, class_counts)
raw_weights = (1.0 - EFFECTIVE_NUMBER_BETA) / np.maximum(effective_numbers, 1e-12)
raw_weights = raw_weights / raw_weights.mean()
used_weights = np.clip(raw_weights, MIN_CLASS_WEIGHT, MAX_CLASS_WEIGHT)
used_weights = used_weights / used_weights.mean()

class_weight = {
    int(index): float(weight)
    for index, weight in zip(classes, used_weights)
}
weight_table = pd.DataFrame({
    'class_name': class_names,
    'train_count': class_counts.astype(int),
    'effective_number_raw_weight': raw_weights,
    'used_weight': used_weights,
})
display(weight_table)
print('Class weight:', class_weight)


## 14. Bangun Model

In [ ]:
# optimizer dan model ResNet50V2
def make_optimizer(learning_rate):
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=WEIGHT_DECAY,
            clipnorm=1.0,
        )
    except Exception:
        return tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        )


def build_model():
    # Augmentasi konservatif: menjaga warna penyakit dan tidak mengecilkan lesi.
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal_and_vertical'),
        tf.keras.layers.RandomRotation(0.06, fill_mode='reflect'),
        tf.keras.layers.RandomTranslation(
            height_factor=0.03,
            width_factor=0.03,
            fill_mode='reflect',
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.08, 0.0),
            width_factor=(-0.08, 0.0),
            fill_mode='reflect',
        ),
        tf.keras.layers.RandomContrast(0.08),
        tf.keras.layers.RandomBrightness(0.04, value_range=(0.0, 255.0)),
    ], name='augmentation')

    base_model = tf.keras.applications.ResNet50V2(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3),
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = augmentation(inputs)
    x = tf.keras.layers.Rescaling(
        1.0 / 127.5,
        offset=-1.0,
        name='resnet_v2_preprocessing',
    )(x)
    feature_map = base_model(x, training=False)

    # GAP mempertahankan konteks, GMP membantu lesi kecil. Backbone tidak diganti.
    average_features = tf.keras.layers.GlobalAveragePooling2D(
        name='global_average_pooling'
    )(feature_map)
    maximum_features = tf.keras.layers.GlobalMaxPooling2D(
        name='global_max_pooling'
    )(feature_map)
    x = tf.keras.layers.Concatenate(name='multi_pooling_features')(
        [average_features, maximum_features]
    )
    x = tf.keras.layers.BatchNormalization(name='head_batch_normalization')(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE, name='head_dropout_1')(x)
    x = tf.keras.layers.Dense(
        DENSE_UNITS,
        activation='gelu',
        kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY),
        name='head_dense',
    )(x)
    x = tf.keras.layers.BatchNormalization(name='head_dense_batch_normalization')(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE * 0.75, name='head_dropout_2')(x)
    outputs = tf.keras.layers.Dense(
        len(class_names),
        activation=None,
        kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY),
        dtype='float32',
        name='logits',
    )(x)
    return tf.keras.Model(inputs, outputs, name='riceleaf_resnet50v2')


def compile_model(model, learning_rate):
    model.compile(
        optimizer=make_optimizer(learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=LABEL_SMOOTHING,
            name='label_smoothed_cross_entropy',
        ),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        ],
    )


model = build_model()
compile_model(model, HEAD_LR)
model.summary()


## 15. Callback

In [ ]:
# robust validation metrics + persistent checkpoint/early stopping
val_true_for_callback = val_df['class_name'].map(class_to_index).astype(int).to_numpy()
hispa_index = class_to_index['Hispa']


def wilson_lower_bound(successes, total, z=WILSON_Z):
    if total <= 0:
        return 0.0
    proportion = successes / total
    denominator = 1.0 + (z * z) / total
    centre = proportion + (z * z) / (2.0 * total)
    adjustment = z * math.sqrt(
        (proportion * (1.0 - proportion) / total)
        + (z * z) / (4.0 * total * total)
    )
    return float((centre - adjustment) / denominator)


def calculate_balanced_selection(labels, predictions):
    per_class_precision, per_class_recall, per_class_f1, support = (
        precision_recall_fscore_support(
            labels,
            predictions,
            labels=np.arange(len(class_names)),
            average=None,
            zero_division=0,
        )
    )
    true_positive = np.array([
        np.sum((labels == index) & (predictions == index))
        for index in range(len(class_names))
    ])
    recall_lcb = np.array([
        wilson_lower_bound(int(tp), int(total))
        for tp, total in zip(true_positive, support)
    ])

    macro_f1 = float(np.mean(per_class_f1))
    min_class_f1 = float(np.min(per_class_f1))
    min_class_recall = float(np.min(per_class_recall))
    min_recall_lcb = float(np.min(recall_lcb))
    geometric_mean_f1 = float(np.exp(np.mean(np.log(np.clip(per_class_f1, 1e-6, 1.0)))))
    selection_score = (
        SELECTION_MACRO_WEIGHT * macro_f1
        + SELECTION_GEOMEAN_WEIGHT * geometric_mean_f1
        + SELECTION_WORST_CLASS_WEIGHT * min_class_f1
        + SELECTION_RECALL_LCB_WEIGHT * min_recall_lcb
    )

    return {
        'macro_f1': macro_f1,
        'min_class_f1': min_class_f1,
        'min_class_recall': min_class_recall,
        'min_recall_lcb': min_recall_lcb,
        'geometric_mean_f1': geometric_mean_f1,
        'hispa_f1': float(per_class_f1[hispa_index]),
        'hispa_recall': float(per_class_recall[hispa_index]),
        'selection_score': float(selection_score),
        'per_class_precision': per_class_precision,
        'per_class_recall': per_class_recall,
        'per_class_recall_lcb': recall_lcb,
        'per_class_f1': per_class_f1,
    }


class ValidationSelectionMetrics(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        validation_logits = self.model.predict(val_ds, verbose=0)
        validation_predictions = np.argmax(validation_logits, axis=1)
        selection = calculate_balanced_selection(
            val_true_for_callback,
            validation_predictions,
        )
        for metric_name in [
            'macro_f1', 'min_class_f1', 'min_class_recall',
            'min_recall_lcb', 'geometric_mean_f1', 'hispa_f1',
            'hispa_recall', 'selection_score',
        ]:
            logs[f'val_{metric_name}'] = selection[metric_name]

        for index, class_name in enumerate(class_names):
            safe_name = class_name.lower()
            logs[f'val_f1_{safe_name}'] = float(selection['per_class_f1'][index])
            logs[f'val_recall_{safe_name}'] = float(selection['per_class_recall'][index])
            logs[f'val_recall_lcb_{safe_name}'] = float(
                selection['per_class_recall_lcb'][index]
            )

        print(
            '\nRobust validation -',
            f"macro F1: {selection['macro_f1']:.4f},",
            f"worst F1: {selection['min_class_f1']:.4f},",
            f"worst recall LCB: {selection['min_recall_lcb']:.4f},",
            f"geomean F1: {selection['geometric_mean_f1']:.4f},",
            f"Hispa F1: {selection['hispa_f1']:.4f},",
            f"score: {selection['selection_score']:.4f}",
        )


class PersistentBestAndEarlyStop(tf.keras.callbacks.Callback):
    """Checkpoint tidak tertimpa saat cell dilanjutkan setelah runtime putus."""

    def __init__(self, checkpoint_path, state_path, initial_epoch=0):
        super().__init__()
        self.checkpoint_path = Path(checkpoint_path)
        self.state_path = Path(state_path)
        self.initial_epoch = int(initial_epoch)
        self.best = -np.inf
        self.best_epoch = -1
        if self.state_path.exists():
            state = json.loads(self.state_path.read_text(encoding='utf-8'))
            self.best = float(state.get('best_score', -np.inf))
            self.best_epoch = int(state.get('best_epoch', -1))
        self.wait = max(0, self.initial_epoch - self.best_epoch - 1)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get('val_selection_score')
        if current is None:
            raise RuntimeError('val_selection_score tidak tersedia.')

        if float(current) > self.best + SELECTION_MIN_DELTA:
            self.best = float(current)
            self.best_epoch = int(epoch)
            self.wait = 0
            self.model.save(self.checkpoint_path)
            self.state_path.write_text(
                json.dumps({
                    'best_score': self.best,
                    'best_epoch': self.best_epoch,
                    'updated_at': datetime.now().isoformat(),
                }, indent=2),
                encoding='utf-8',
            )
            print(f'Checkpoint robust tersimpan: epoch {epoch + 1}, score {self.best:.4f}')
        else:
            self.wait += 1
            if self.wait >= EARLY_STOPPING_PATIENCE:
                print(
                    f'Early stop: tidak ada peningkatan robust selama {self.wait} epoch. '
                    f'Best epoch={self.best_epoch + 1}, score={self.best:.4f}'
                )
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.checkpoint_path.exists():
            best_model = tf.keras.models.load_model(self.checkpoint_path, compile=False)
            self.model.set_weights(best_model.get_weights())
            print('Bobot checkpoint robust terbaik dipulihkan.')


def completed_epochs(history_path):
    history_path = Path(history_path)
    if not history_path.exists():
        return 0
    try:
        return int(len(pd.read_csv(history_path)))
    except Exception:
        return 0


def create_callbacks(checkpoint_path, history_path, state_path, initial_epoch):
    history_path = Path(history_path)
    return [
        ValidationSelectionMetrics(),
        PersistentBestAndEarlyStop(
            checkpoint_path,
            state_path,
            initial_epoch=initial_epoch,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_selection_score',
            mode='max',
            factor=0.50,
            patience=LR_PATIENCE,
            min_lr=1e-7,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(history_path, append=history_path.exists()),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


STAGE1_MODEL = CHECKPOINT_DIR / 'best_stage1.keras'
STAGE2_MODEL = CHECKPOINT_DIR / 'best_stage2.keras'
STAGE1_STATE = CHECKPOINT_DIR / 'best_stage1_state.json'
STAGE2_STATE = CHECKPOINT_DIR / 'best_stage2_state.json'
STAGE1_HISTORY = RESULT_DIR / 'history_stage1.csv'
STAGE2_HISTORY = RESULT_DIR / 'history_stage2.csv'
STAGE1_MARKER = CHECKPOINT_DIR / 'stage1_complete.json'
STAGE2_MARKER = CHECKPOINT_DIR / 'stage2_complete.json'


## 16. Training Tahap 1

In [ ]:
# training head; aman dilanjutkan dari epoch terakhir yang tercatat
if FORCE_RETRAIN or not STAGE1_MARKER.exists() or not STAGE1_MODEL.exists():
    stage1_initial_epoch = 0 if FORCE_RETRAIN else completed_epochs(STAGE1_HISTORY)
    if STAGE1_MODEL.exists() and stage1_initial_epoch > 0 and not FORCE_RETRAIN:
        model = tf.keras.models.load_model(STAGE1_MODEL)
        print('Melanjutkan stage 1 dari checkpoint:', STAGE1_MODEL)
    else:
        stage1_initial_epoch = 0

    model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=stage1_initial_epoch,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=create_callbacks(
            STAGE1_MODEL,
            STAGE1_HISTORY,
            STAGE1_STATE,
            stage1_initial_epoch,
        ),
    )
    STAGE1_MARKER.write_text(
        json.dumps({
            'completed_at': datetime.now().isoformat(),
            'completed_epochs': completed_epochs(STAGE1_HISTORY),
        }, indent=2),
        encoding='utf-8',
    )
else:
    model = tf.keras.models.load_model(STAGE1_MODEL)
    print('Tahap 1 sudah tersedia.')


## 17. Fine-Tuning

In [ ]:
# fine-tuning ResNet50V2; BatchNorm tetap frozen untuk batch kecil
if FORCE_RETRAIN or not STAGE2_MARKER.exists() or not STAGE2_MODEL.exists():
    stage2_initial_epoch = 0 if FORCE_RETRAIN else completed_epochs(STAGE2_HISTORY)

    if STAGE2_MODEL.exists() and stage2_initial_epoch > 0 and not FORCE_RETRAIN:
        model = tf.keras.models.load_model(STAGE2_MODEL)
        print('Melanjutkan stage 2 dari checkpoint:', STAGE2_MODEL)
    else:
        stage2_initial_epoch = 0
        model = tf.keras.models.load_model(STAGE1_MODEL)
        base_model = model.get_layer('resnet50v2')
        base_model.trainable = True
        for layer in base_model.layers:
            layer.trainable = False
        for layer in base_model.layers[-FINE_TUNE_LAYERS:]:
            layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)
        compile_model(model, FINE_TUNE_LR)

    base_model = model.get_layer('resnet50v2')
    trainable_backbone_layers = sum(int(layer.trainable) for layer in base_model.layers)
    print('Trainable backbone layers:', trainable_backbone_layers, '/', len(base_model.layers))

    model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=stage2_initial_epoch,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=create_callbacks(
            STAGE2_MODEL,
            STAGE2_HISTORY,
            STAGE2_STATE,
            stage2_initial_epoch,
        ),
    )
    STAGE2_MARKER.write_text(
        json.dumps({
            'completed_at': datetime.now().isoformat(),
            'completed_epochs': completed_epochs(STAGE2_HISTORY),
            'fine_tune_layers': FINE_TUNE_LAYERS,
            'image_size': list(IMAGE_SIZE),
        }, indent=2),
        encoding='utf-8',
    )
else:
    print('Tahap 2 sudah tersedia.')


## 18. Pilih Model Terbaik

In [ ]:
# pilih stage hanya dari validation dengan objective robust yang sama seperti checkpoint
def evaluate_candidate(candidate_model, stage_name):
    evaluation = candidate_model.evaluate(val_ds, verbose=0, return_dict=True)
    validation_logits = candidate_model.predict(val_ds, verbose=0)
    validation_predictions = np.argmax(validation_logits, axis=1)
    selection = calculate_balanced_selection(
        val_true_for_callback,
        validation_predictions,
    )
    return {
        'stage': stage_name,
        **evaluation,
        'macro_f1': selection['macro_f1'],
        'min_class_f1': selection['min_class_f1'],
        'min_class_recall': selection['min_class_recall'],
        'min_recall_lcb': selection['min_recall_lcb'],
        'geometric_mean_f1': selection['geometric_mean_f1'],
        'hispa_f1': selection['hispa_f1'],
        'hispa_recall': selection['hispa_recall'],
        'selection_score': selection['selection_score'],
    }


stage1_model = tf.keras.models.load_model(STAGE1_MODEL)
stage2_model = tf.keras.models.load_model(STAGE2_MODEL)
candidate_rows = [
    evaluate_candidate(stage1_model, 'stage1'),
    evaluate_candidate(stage2_model, 'stage2'),
]
comparison_df = pd.DataFrame(candidate_rows)
display(comparison_df.sort_values('selection_score', ascending=False))

best_candidate_index = comparison_df['selection_score'].idxmax()
selected_stage = comparison_df.loc[best_candidate_index, 'stage']
selected_path = STAGE2_MODEL if selected_stage == 'stage2' else STAGE1_MODEL

FINAL_MODEL_PATH = MODEL_DIR / 'best_model.keras'
shutil.copy2(selected_path, FINAL_MODEL_PATH)
final_model = tf.keras.models.load_model(FINAL_MODEL_PATH, compile=False)

print('Model terpilih:', selected_stage)
print('Selection score:', round(float(comparison_df.loc[best_candidate_index, 'selection_score']), 4))
print('Model final:', FINAL_MODEL_PATH)


## 19. Grafik Training

In [ ]:
# kurva training
history_frames = []
for stage_name, history_path in [
    ('stage1', STAGE1_HISTORY),
    ('stage2', STAGE2_HISTORY),
]:
    if history_path.exists():
        frame = pd.read_csv(history_path)
        frame['stage'] = stage_name
        history_frames.append(frame)

if not history_frames:
    raise FileNotFoundError(
        'Riwayat training tidak ditemukan.'
    )

history_df = pd.concat(
    history_frames,
    ignore_index=True,
)
history_df['global_epoch'] = np.arange(
    1,
    len(history_df) + 1,
)
history_df.to_csv(
    RESULT_DIR / 'training_history.csv',
    index=False,
)

for metric, title, filename in [
    (
        'accuracy',
        'Training dan Validation Accuracy',
        'accuracy_curve.png',
    ),
    (
        'loss',
        'Training dan Validation Loss',
        'loss_curve.png',
    ),
]:
    plt.figure(figsize=(9, 5))
    plt.plot(
        history_df['global_epoch'],
        history_df[metric],
        label='train',
    )
    plt.plot(
        history_df['global_epoch'],
        history_df[f'val_{metric}'],
        label='validation',
    )
    plt.xlabel('Epoch')
    plt.ylabel(metric.title())
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RESULT_DIR / filename,
        dpi=200,
        bbox_inches='tight',
    )
    plt.show()

if {
    'val_macro_f1',
    'val_min_class_f1',
    'val_min_class_recall',
    'val_geometric_mean_f1',
    'val_hispa_f1',
    'val_hispa_recall',
    'val_selection_score',
}.issubset(history_df.columns):
    plt.figure(figsize=(9, 5))
    for metric in [
        'val_macro_f1',
        'val_min_class_f1',
        'val_min_class_recall',
        'val_geometric_mean_f1',
        'val_hispa_f1',
        'val_hispa_recall',
        'val_selection_score',
    ]:
        plt.plot(
            history_df['global_epoch'],
            history_df[metric],
            label=metric,
        )
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.ylim(0, 1)
    plt.title(
        'Validation Balanced Selection Metrics'
    )
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RESULT_DIR
        / 'selection_metrics_curve.png',
        dpi=200,
        bbox_inches='tight',
    )
    plt.show()

# Gunakan epoch terbaik menurut target seleksi yang sama
# dengan checkpoint, bukan val_loss atau epoch terakhir.
if 'val_selection_score' in history_df.columns:
    best_history_index = (
        history_df['val_selection_score'].idxmax()
    )
else:
    best_history_index = (
        history_df['val_loss'].idxmin()
    )

best_history_row = history_df.loc[
    best_history_index
]
generalization_gap = float(
    best_history_row['accuracy']
    - best_history_row['val_accuracy']
)

print(
    'Best history stage:',
    best_history_row['stage'],
)
print(
    'Best history epoch:',
    int(best_history_row['global_epoch']),
)
if 'val_macro_f1' in best_history_row:
    print(
        'Best validation macro F1:',
        round(
            float(
                best_history_row['val_macro_f1']
            ),
            4,
        ),
    )
    print(
        'Best validation minimum class F1:',
        round(float(best_history_row['val_min_class_f1']), 4),
    )
    print(
        'Best validation minimum class recall:',
        round(float(best_history_row['val_min_class_recall']), 4),
    )
    print(
        'Best validation geometric mean F1:',
        round(float(best_history_row['val_geometric_mean_f1']), 4),
    )
    print(
        'Best validation Hispa F1:',
        round(float(best_history_row['val_hispa_f1']), 4),
    )
    print(
        'Best validation Hispa recall:',
        round(float(best_history_row['val_hispa_recall']), 4),
    )
print(
    'Generalization gap:',
    round(generalization_gap, 4),
)

if generalization_gap > 0.10:
    print('Indikasi overfitting perlu diperiksa.')
elif generalization_gap < -0.08:
    print(
        'Validation lebih tinggi karena augmentasi hanya aktif pada train; '
        'ini bukan bukti underfitting dengan sendirinya.'
    )
else:
    print('Kurva perlu dibaca bersama hasil test per kelas.')

## 20. Pilih TTA dan Kalibrasi Confidence

TTA, temperature, dan threshold dipilih menggunakan validation. Test belum digunakan pada tahap ini.


In [ ]:
# pilih TTA pada validation dengan toleransi: jika score hampir seri, pilih worst-class F1 lebih tinggi
def predict_logits_with_tta(model, dataset, use_tta=True):
    all_logits = []
    for images, _ in dataset:
        batch_logits = [model(images, training=False).numpy()]
        if use_tta:
            batch_logits.extend([
                model(tf.image.flip_left_right(images), training=False).numpy(),
                model(tf.image.flip_up_down(images), training=False).numpy(),
            ])
        all_logits.append(np.mean(batch_logits, axis=0))
    return np.concatenate(all_logits, axis=0)


val_true = val_df['class_name'].map(class_to_index).astype(int).to_numpy()

tta_options = [False, True] if AUTO_SELECT_TTA else [bool(USE_TTA)]
tta_logits_cache = {}
tta_rows = []
for use_tta_candidate in tta_options:
    candidate_logits = predict_logits_with_tta(
        final_model,
        val_ds,
        use_tta=use_tta_candidate,
    )
    tta_logits_cache[use_tta_candidate] = candidate_logits
    candidate_predictions = np.argmax(candidate_logits, axis=1)
    selection = calculate_balanced_selection(val_true, candidate_predictions)
    tta_rows.append({
        'use_tta': bool(use_tta_candidate),
        'macro_f1': selection['macro_f1'],
        'min_class_f1': selection['min_class_f1'],
        'min_class_recall': selection['min_class_recall'],
        'min_recall_lcb': selection['min_recall_lcb'],
        'geometric_mean_f1': selection['geometric_mean_f1'],
        'hispa_f1': selection['hispa_f1'],
        'hispa_recall': selection['hispa_recall'],
        'selection_score': selection['selection_score'],
    })

tta_comparison_df = pd.DataFrame(tta_rows)
best_tta_score = float(tta_comparison_df['selection_score'].max())
tta_eligible_df = tta_comparison_df[
    tta_comparison_df['selection_score'] >= best_tta_score - TTA_SELECTION_TOLERANCE
].copy()
tta_eligible_df = tta_eligible_df.sort_values(
    ['min_class_f1', 'min_recall_lcb', 'geometric_mean_f1', 'macro_f1'],
    ascending=False,
)
USE_TTA = bool(tta_eligible_df.iloc[0]['use_tta'])
display(tta_comparison_df.sort_values('selection_score', ascending=False))

val_logits = tta_logits_cache[USE_TTA]

def temperature_nll(temperature, logits, labels):
    probabilities = softmax(logits / temperature, axis=1)
    return log_loss(labels, probabilities, labels=np.arange(len(class_names)))


result = minimize_scalar(
    temperature_nll,
    bounds=(0.25, 10.0),
    method='bounded',
    args=(val_logits, val_true),
)
TEMPERATURE = float(result.x)
val_probabilities_raw = softmax(val_logits, axis=1)
val_probabilities = softmax(val_logits / TEMPERATURE, axis=1)

print('TTA selected:', USE_TTA)
print('Temperature:', TEMPERATURE)
print('Validation NLL sebelum:', log_loss(val_true, val_probabilities_raw))
print('Validation NLL sesudah:', log_loss(val_true, val_probabilities))


## 21. Reliability Diagram

In [ ]:
# expected calibration error; tahap ini hanya memakai validation
def expected_calibration_error(labels, probabilities, bins=10):
    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correctness = (predictions == labels).astype(float)
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    ece, rows = 0.0, []
    for index in range(bins):
        lower, upper = boundaries[index], boundaries[index + 1]
        mask = (
            (confidences >= lower)
            & (confidences <= upper if index == bins - 1 else confidences < upper)
        )
        if not np.any(mask):
            continue
        accuracy_bin = float(correctness[mask].mean())
        confidence_bin = float(confidences[mask].mean())
        fraction = float(mask.mean())
        ece += abs(accuracy_bin - confidence_bin) * fraction
        rows.append({
            'accuracy': accuracy_bin,
            'confidence': confidence_bin,
            'fraction': fraction,
        })
    return float(ece), pd.DataFrame(rows)


val_ece_before, val_calibration_before = expected_calibration_error(
    val_true,
    val_probabilities_raw,
)
val_ece_after, val_calibration_after = expected_calibration_error(
    val_true,
    val_probabilities,
)
print('Validation ECE sebelum:', val_ece_before)
print('Validation ECE sesudah:', val_ece_after)

plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], linestyle='--', label='ideal')
plt.plot(
    val_calibration_before['confidence'],
    val_calibration_before['accuracy'],
    marker='o',
    label='sebelum',
)
plt.plot(
    val_calibration_after['confidence'],
    val_calibration_after['accuracy'],
    marker='o',
    label='sesudah',
)
plt.xlabel('Rata-rata Confidence')
plt.ylabel('Accuracy')
plt.title('Validation Reliability Diagram')
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / 'validation_reliability_diagram.png',
    dpi=200,
    bbox_inches='tight',
)
plt.show()


## 22. Tentukan Threshold

In [ ]:
# threshold validation: confidence + margin; simpan point estimate dan Wilson lower bound
val_predictions = val_probabilities.argmax(axis=1)
val_confidences = val_probabilities.max(axis=1)
val_sorted_probabilities = np.sort(val_probabilities, axis=1)
val_margins = val_sorted_probabilities[:, -1] - val_sorted_probabilities[:, -2]

threshold_rows = []
for threshold in np.arange(LOW_CONF_THRESHOLD, 0.96, 0.01):
    mask = (
        (val_confidences >= threshold)
        & (val_margins >= MIN_CONFIDENCE_MARGIN)
    )
    accepted_count = int(mask.sum())
    correct_count = int((val_predictions[mask] == val_true[mask]).sum()) if accepted_count else 0
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'min_margin': MIN_CONFIDENCE_MARGIN,
        'coverage': float(mask.mean()),
        'accepted_count': accepted_count,
        'accepted_accuracy': correct_count / accepted_count if accepted_count else np.nan,
        'accepted_accuracy_lcb90': wilson_lower_bound(correct_count, accepted_count),
    })

threshold_df = pd.DataFrame(threshold_rows)
qualified = threshold_df[
    (threshold_df['accepted_accuracy'] >= TARGET_ACCEPTED_ACCURACY)
    & (threshold_df['coverage'] >= MIN_ACCEPTED_COVERAGE)
    & (threshold_df['accepted_count'] >= MIN_ACCEPTED_COUNT)
    & (threshold_df['threshold'] > LOW_CONF_THRESHOLD)
]

HIGH_CONF_THRESHOLD = float(qualified.iloc[0]['threshold']) if len(qualified) else 0.80
HIGH_CONF_THRESHOLD = max(HIGH_CONF_THRESHOLD, LOW_CONF_THRESHOLD + 0.10)
threshold_df.to_csv(RESULT_DIR / 'threshold_analysis.csv', index=False)
display(threshold_df)

print('Low threshold:', LOW_CONF_THRESHOLD)
print('High threshold:', HIGH_CONF_THRESHOLD)
print('Minimum top-1 margin:', MIN_CONFIDENCE_MARGIN)
print('Threshold dipilih dari validation saja.')


## 23. Evaluasi Final Test

Mulai sel ini, test set baru dibuka. Jangan mengubah hyperparameter berdasarkan hasil bagian test lalu mengklaimnya sebagai evaluasi unbiased. Jika v8 direvisi lagi, gunakan holdout eksternal baru untuk klaim final.


In [ ]:
# buka final test: tidak ada keputusan training/selection setelah titik ini
test_true = test_df['class_name'].map(class_to_index).astype(int).to_numpy()
test_logits = predict_logits_with_tta(final_model, test_ds, use_tta=USE_TTA)
test_probabilities_raw = softmax(test_logits, axis=1)
test_probabilities = softmax(test_logits / TEMPERATURE, axis=1)
ece_before, test_calibration_before = expected_calibration_error(
    test_true,
    test_probabilities_raw,
)
ece_after, test_calibration_after = expected_calibration_error(
    test_true,
    test_probabilities,
)
# metrik test
test_predictions = test_probabilities.argmax(
    axis=1
)
test_confidences = test_probabilities.max(
    axis=1
)
test_sorted_probabilities = np.sort(
    test_probabilities,
    axis=1,
)
test_margins = (
    test_sorted_probabilities[:, -1]
    - test_sorted_probabilities[:, -2]
)

test_balanced_selection = calculate_balanced_selection(
    test_true,
    test_predictions,
)

accuracy = accuracy_score(
    test_true,
    test_predictions,
)
macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        test_true,
        test_predictions,
        average='macro',
        zero_division=0,
    )
)
weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        test_true,
        test_predictions,
        average='weighted',
        zero_division=0,
    )
)

(
    hispa_precision_array,
    hispa_recall_array,
    hispa_f1_array,
    hispa_support_array,
) = precision_recall_fscore_support(
    test_true,
    test_predictions,
    labels=[hispa_index],
    average=None,
    zero_division=0,
)
hispa_precision = float(
    hispa_precision_array[0]
)
hispa_recall = float(
    hispa_recall_array[0]
)
hispa_f1 = float(
    hispa_f1_array[0]
)
hispa_support = int(
    hispa_support_array[0]
)

test_one_hot = label_binarize(
    test_true,
    classes=np.arange(len(class_names)),
)
macro_roc_auc = roc_auc_score(
    test_one_hot,
    test_probabilities,
    average='macro',
    multi_class='ovr',
)
macro_pr_auc = average_precision_score(
    test_one_hot,
    test_probabilities,
    average='macro',
)
multiclass_brier = float(
    np.mean(
        np.sum(
            (
                test_probabilities
                - test_one_hot
            ) ** 2,
            axis=1,
        )
    )
)

# Accepted harus sama dengan status `valid` pada API: high threshold + margin.
accepted_mask = (
    (
        test_confidences
        >= HIGH_CONF_THRESHOLD
    )
    & (
        test_margins
        >= MIN_CONFIDENCE_MARGIN
    )
)
accepted_coverage = float(
    accepted_mask.mean()
)
accepted_accuracy = (
    float(
        (
            test_predictions[accepted_mask]
            == test_true[accepted_mask]
        ).mean()
    )
    if accepted_mask.any()
    else np.nan
)

# Reviewable mencakup status valid + uncertain, tetapi bukan low-confidence.
reviewable_mask = (
    (
        test_confidences
        >= LOW_CONF_THRESHOLD
    )
    & (
        test_margins
        >= MIN_CONFIDENCE_MARGIN
    )
)
reviewable_coverage = float(
    reviewable_mask.mean()
)
reviewable_accuracy = (
    float(
        (
            test_predictions[reviewable_mask]
            == test_true[reviewable_mask]
        ).mean()
    )
    if reviewable_mask.any()
    else np.nan
)

metrics = {
    'selected_stage': selected_stage,
    'architecture': 'ResNet50V2',
    'image_size': list(IMAGE_SIZE),
    'tta_enabled': bool(USE_TTA),
    'accuracy': float(accuracy),
    'macro_precision': float(
        macro_precision
    ),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'minimum_class_f1': test_balanced_selection['min_class_f1'],
    'minimum_class_recall': test_balanced_selection['min_class_recall'],
    'geometric_mean_f1': test_balanced_selection['geometric_mean_f1'],
    'hispa_precision': hispa_precision,
    'hispa_recall': hispa_recall,
    'hispa_f1': hispa_f1,
    'hispa_support': hispa_support,
    'weighted_precision': float(
        weighted_precision
    ),
    'weighted_recall': float(
        weighted_recall
    ),
    'weighted_f1': float(weighted_f1),
    'macro_roc_auc_ovr': float(
        macro_roc_auc
    ),
    'macro_pr_auc': float(macro_pr_auc),
    'multiclass_brier': (
        multiclass_brier
    ),
    'ece_before_calibration': float(
        ece_before
    ),
    'ece_after_calibration': float(
        ece_after
    ),
    'temperature': TEMPERATURE,
    'low_conf_threshold': (
        LOW_CONF_THRESHOLD
    ),
    'high_conf_threshold': (
        HIGH_CONF_THRESHOLD
    ),
    'min_confidence_margin': (
        MIN_CONFIDENCE_MARGIN
    ),
    'accepted_coverage': (
        accepted_coverage
    ),
    'accepted_accuracy': (
        accepted_accuracy
    ),
    'reviewable_coverage': (
        reviewable_coverage
    ),
    'reviewable_accuracy': (
        reviewable_accuracy
    ),
    'generalization_gap_selected_epoch': (
        generalization_gap
    ),
}

display(pd.DataFrame([metrics]))

with (
    RESULT_DIR / 'test_metrics.json'
).open('w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)# provenance v8; tulis ulang metrics JSON setelah metadata tambahan dimasukkan
metrics.update({
    'minimum_recall_lcb90': test_balanced_selection['min_recall_lcb'],
    'split_seed': SPLIT_SEED,
    'split_fingerprint': split_fingerprint,
    'test_used_for_selection': False,
    'loss': 'categorical_crossentropy_label_smoothing',
    'label_smoothing': LABEL_SMOOTHING,
})
with (RESULT_DIR / 'test_metrics.json').open('w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)


## 24. Classification Report

In [ ]:
# classification report
report = classification_report(test_true, test_predictions, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv(RESULT_DIR / 'classification_report.csv')
with (RESULT_DIR / 'classification_report.json').open('w', encoding='utf-8') as file:
    json.dump(report, file, indent=2)

## 25. Confusion Matrix

In [ ]:
# confusion matrix
matrix = confusion_matrix(test_true, test_predictions, labels=np.arange(len(class_names)))
normalized_matrix = confusion_matrix(test_true, test_predictions, labels=np.arange(len(class_names)), normalize='true')
pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(RESULT_DIR / 'confusion_matrix.csv')
pd.DataFrame(normalized_matrix, index=class_names, columns=class_names).to_csv(RESULT_DIR / 'confusion_matrix_normalized.csv')

for current_matrix, title, filename, value_format in [
    (matrix, 'Confusion Matrix', 'confusion_matrix.png', 'd'),
    (normalized_matrix, 'Normalized Confusion Matrix', 'confusion_matrix_normalized.png', '.2f'),
]:
    figure, axis = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay(current_matrix, display_labels=class_names).plot(ax=axis, values_format=value_format)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / filename, dpi=200, bbox_inches='tight')
    plt.show()

## 26. Analisis Prediksi Salah

In [ ]:
# analisis kesalahan dengan confidence tinggi
test_result_df = (
    test_df[
        ['path', 'relative_path', 'class_name']
    ]
    .copy()
    .reset_index(drop=True)
)
test_result_df['predicted_class'] = [
    class_names[index]
    for index in test_predictions
]
test_result_df['confidence'] = test_confidences
test_result_df['margin'] = test_margins
test_result_df['correct'] = (
    test_true == test_predictions
)

wrong_df = (
    test_result_df[
        ~test_result_df['correct']
    ]
    .sort_values('confidence', ascending=False)
    .head(12)
)
test_result_df.to_csv(
    RESULT_DIR / 'test_predictions.csv',
    index=False,
)

if wrong_df.empty:
    print('Tidak ada prediksi salah pada test set.')
else:
    columns = 4
    rows = math.ceil(len(wrong_df) / columns)
    plt.figure(figsize=(16, 4 * rows))

    for plot_index, (_, row) in enumerate(
        wrong_df.iterrows(),
        start=1,
    ):
        with Image.open(row['path']) as image:
            image = image.convert('RGB')

        plt.subplot(rows, columns, plot_index)
        plt.imshow(image)
        plt.title(
            f"True: {row['class_name']}\n"
            f"Pred: {row['predicted_class']}\n"
            f"Conf: {row['confidence']:.2%} | "
            f"Margin: {row['margin']:.2%}"
        )
        plt.axis('off')

    plt.tight_layout()
    plt.savefig(
        RESULT_DIR / 'high_confidence_errors.png',
        dpi=200,
        bbox_inches='tight',
    )
    plt.show()

## 27. Simpan Konfigurasi Web

In [ ]:
# konfigurasi web dan metadata eksperimen v8
class_config = {
    'class_names': class_names,
    'class_to_index': class_to_index,
}
inference_config = {
    'image_size': list(IMAGE_SIZE),
    'temperature': TEMPERATURE,
    'low_conf_threshold': LOW_CONF_THRESHOLD,
    'high_conf_threshold': HIGH_CONF_THRESHOLD,
    'min_confidence_margin': MIN_CONFIDENCE_MARGIN,
    'use_tta': bool(USE_TTA),
    'auto_select_tta': bool(AUTO_SELECT_TTA),
    'tta_transforms': ['original', 'horizontal_flip', 'vertical_flip'],
    'resize_method': 'tf.image.resize_with_pad',
    'model_output': 'logits',
    'warning': (
        'Confidence adalah estimasi model, bukan diagnosis pasti. '
        'Low confidence juga bukan bukti bahwa gambar bukan daun padi.'
    ),
}
model_metadata = {
    'project_name': PROJECT_NAME,
    'dataset_handle': DATASET_HANDLE,
    'architecture': 'ResNet50V2',
    'pretrained_weights': 'ImageNet',
    'image_size': list(IMAGE_SIZE),
    'feature_pooling': ['GlobalAveragePooling2D', 'GlobalMaxPooling2D'],
    'dense_units': DENSE_UNITS,
    'fine_tune_layers': FINE_TUNE_LAYERS,
    'fine_tune_epochs_max': FINE_TUNE_EPOCHS,
    'loss': 'CategoricalCrossentropy(from_logits=True)',
    'label_smoothing': LABEL_SMOOTHING,
    'class_balance_strategy': 'effective_number_class_weight_only',
    'effective_number_beta': EFFECTIVE_NUMBER_BETA,
    'class_weights': class_weight,
    'oversampling_enabled': False,
    'selection_metric': (
        f'{SELECTION_MACRO_WEIGHT:.2f} * macro_f1 + '
        f'{SELECTION_GEOMEAN_WEIGHT:.2f} * geometric_mean_f1 + '
        f'{SELECTION_WORST_CLASS_WEIGHT:.2f} * min_class_f1 + '
        f'{SELECTION_RECALL_LCB_WEIGHT:.2f} * min_recall_lcb90'
    ),
    'selection_wilson_z': WILSON_Z,
    'classes': class_names,
    'selected_stage': selected_stage,
    'tensorflow_version': tf.__version__,
    'unique_images': int(len(clean_df)),
    'train_images': int(len(train_df)),
    'validation_images': int(len(val_df)),
    'test_images': int(len(test_df)),
    'split_seed': SPLIT_SEED,
    'split_fingerprint': split_fingerprint,
    'phash_group_distance': PHASH_DISTANCE,
    'test_used_for_selection': False,
    'created_at': datetime.now().isoformat(),
    'test_metrics': metrics,
}
api_contract = {
    'endpoint': '/api/predict',
    'method': 'POST',
    'content_type': 'multipart/form-data',
    'image_field': 'image',
    'allowed_extensions': ['jpg', 'jpeg', 'png'],
    'max_file_size_mb': 5,
    'response': {
        'status': 'valid | uncertain | rejected_low_confidence',
        'predicted_class': 'string | null',
        'top_class_internal': 'string',
        'confidence': 'float',
        'margin': 'float',
        'scores': {name: 'float' for name in class_names},
    },
}
disease_information = {
    name: {
        'display_name': name,
        'description': '',
        'treatment': '',
        'prevention': '',
        'review_status': 'Isi harus ditinjau ahli pertanian.',
    }
    for name in class_names
}

for filename, content in [
    ('class_names.json', class_config),
    ('inference_config.json', inference_config),
    ('model_metadata.json', model_metadata),
    ('api_contract.json', api_contract),
    ('disease_information.json', disease_information),
]:
    with (MODEL_DIR / filename).open('w', encoding='utf-8') as file:
        json.dump(content, file, indent=2, ensure_ascii=False)

print('Konfigurasi v8 tersimpan.')


## 28. Inference Service Flask

In [ ]:
# buat service Flask
inference_lines = [
    'from pathlib import Path',
    'import json',
    'import numpy as np',
    'import tensorflow as tf',
    (
        'from PIL import Image, '
        'UnidentifiedImageError'
    ),
    '',
    "BUNDLE_DIR = Path(__file__).resolve().parent",
    (
        "model = tf.keras.models.load_model("
        "BUNDLE_DIR / 'best_model.keras', "
        "compile=False)"
    ),
    (
        "class_config = json.loads("
        "(BUNDLE_DIR / 'class_names.json')"
        ".read_text(encoding='utf-8'))"
    ),
    (
        "config = json.loads("
        "(BUNDLE_DIR / 'inference_config.json')"
        ".read_text(encoding='utf-8'))"
    ),
    "class_names = class_config['class_names']",
    "image_size = tuple(config['image_size'])",
    "temperature = float(config['temperature'])",
    (
        "low_threshold = float("
        "config['low_conf_threshold'])"
    ),
    (
        "high_threshold = float("
        "config['high_conf_threshold'])"
    ),
    (
        "min_margin = float(config.get("
        "'min_confidence_margin', 0.10))"
    ),
    (
        "use_tta = bool(config.get("
        "'use_tta', True))"
    ),
    '',
    'def prepare_image(file_source):',
    '    try:',
    (
        "        image = Image.open("
        "file_source).convert('RGB')"
    ),
    (
        '    except '
        '(UnidentifiedImageError, OSError) '
        'as error:'
    ),
    (
        "        raise ValueError("
        "'File bukan gambar yang valid.') "
        "from error"
    ),
    '    image = np.asarray(image)',
    '    image = tf.image.resize_with_pad(',
    (
        '        image, image_size[0], '
        'image_size[1], antialias=True'
    ),
    '    )',
    '    image = tf.cast(image, tf.float32)',
    (
        '    return tf.expand_dims('
        'image, axis=0)'
    ),
    '',
    'def predict_image(file_source):',
    (
        '    image_batch = '
        'prepare_image(file_source)'
    ),
    '    variants = [image_batch]',
    '    if use_tta:',
    '        variants.extend([',
    (
        '            tf.image.'
        'flip_left_right(image_batch),'
    ),
    (
        '            tf.image.'
        'flip_up_down(image_batch),'
    ),
    '        ])',
    (
        '    variant_batch = '
        'tf.concat(variants, axis=0)'
    ),
    (
        '    variant_logits = model.predict('
        'variant_batch, verbose=0)'
    ),
    (
        '    logits = np.mean('
        'variant_logits, axis=0)'
    ),
    (
        '    probabilities = tf.nn.softmax('
        'logits / temperature).numpy()'
    ),
    (
        '    predicted_index = int('
        'np.argmax(probabilities))'
    ),
    (
        '    confidence = float('
        'probabilities[predicted_index])'
    ),
    (
        '    sorted_probabilities = '
        'np.sort(probabilities)'
    ),
    (
        '    margin = float('
        'sorted_probabilities[-1] - '
        'sorted_probabilities[-2])'
    ),
    (
        '    top_class = '
        'class_names[predicted_index]'
    ),
    '    if confidence < low_threshold:',
    (
        "        status, predicted_class = "
        "'rejected_low_confidence', None"
    ),
    (
        '    elif confidence < high_threshold '
        'or margin < min_margin:'
    ),
    (
        "        status, predicted_class = "
        "'uncertain', top_class"
    ),
    '    else:',
    (
        "        status, predicted_class = "
        "'valid', top_class"
    ),
    '    scores = {',
    '        name: float(score)',
    (
        '        for name, score in '
        'zip(class_names, probabilities)'
    ),
    '    }',
    '    return {',
    "        'status': status,",
    (
        "        'predicted_class': "
        "predicted_class,"
    ),
    (
        "        'top_class_internal': "
        "top_class,"
    ),
    "        'confidence': confidence,",
    "        'margin': margin,",
    "        'scores': scores,",
    '    }',
]

api_lines = [
    (
        'from flask import Flask, jsonify, '
        'request'
    ),
    (
        'from inference_service import '
        'predict_image'
    ),
    '',
    'app = Flask(__name__)',
    (
        "app.config['MAX_CONTENT_LENGTH'] "
        '= 5 * 1024 * 1024'
    ),
    (
        "ALLOWED_EXTENSIONS = "
        "{'jpg', 'jpeg', 'png'}"
    ),
    '',
    'def allowed_file(filename):',
    (
        "    return '.' in filename and "
        "filename.rsplit('.', 1)[1].lower() "
        "in ALLOWED_EXTENSIONS"
    ),
    '',
    "@app.post('/api/predict')",
    'def predict():',
    "    if 'image' not in request.files:",
    (
        "        return jsonify({'error': "
        "'Field image wajib diisi.'}), 400"
    ),
    (
        "    image_file = "
        "request.files['image']"
    ),
    (
        '    if not image_file.filename '
        'or not allowed_file('
        'image_file.filename):'
    ),
    (
        "        return jsonify({'error': "
        "'Format gambar tidak didukung.'}), "
        "400"
    ),
    '    try:',
    (
        '        result = predict_image('
        'image_file.stream)'
    ),
    '    except ValueError as error:',
    (
        "        return jsonify({'error': "
        "str(error)}), 400"
    ),
    '    return jsonify(result), 200',
    '',
    "if __name__ == '__main__':",
    (
        "    app.run(host='0.0.0.0', "
        'port=5000, debug=False)'
    ),
]

(MODEL_DIR / 'inference_service.py').write_text(
    '\n'.join(inference_lines) + '\n',
    encoding='utf-8',
)
(MODEL_DIR / 'flask_api_example.py').write_text(
    '\n'.join(api_lines) + '\n',
    encoding='utf-8',
)

requirements = [
    f'tensorflow=={tf.__version__}',
    'Flask>=3.0,<4.0',
    'Pillow>=10.0',
    'numpy>=1.26',
]
(MODEL_DIR / 'requirements_web.txt').write_text(
    '\n'.join(requirements) + '\n',
    encoding='utf-8',
)

print('Service Flask tersimpan.')

## 29. Model Card

In [ ]:
# model card v8
model_card_lines = [
    '# RiceLeafs ResNet50V2 v8 Model Card',
    '',
    '## Dataset dan Split',
    f'- Sumber: {DATASET_HANDLE}',
    f'- Data unik: {len(clean_df)}',
    f'- Train: {len(train_df)}',
    f'- Validation: {len(val_df)}',
    f'- Test: {len(test_df)}',
    f'- Split seed: {SPLIT_SEED}',
    f'- Split fingerprint: {split_fingerprint}',
    f"- Kelas: {', '.join(class_names)}",
    '- Exact duplicate dihapus; near-duplicate satu kelas dikelompokkan sebelum split.',
    '- Kandidat near-duplicate lintas kelas hanya dilaporkan untuk audit manual.',
    '',
    '## Model',
    '- Backbone: ResNet50V2, bobot awal ImageNet',
    f'- Input: {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]} RGB',
    '- Feature head: GAP + GMP + Dense',
    f'- Fine-tuned layers maksimum: {FINE_TUNE_LAYERS}',
    '- Output: logits',
    f'- TTA terpilih dari validation: {USE_TTA}',
    f'- Temperature: {TEMPERATURE:.6f}',
    '',
    '## Strategi Training v8',
    f'- Loss: categorical cross-entropy, label smoothing {LABEL_SMOOTHING:.2f}',
    f'- Effective-number beta: {EFFECTIVE_NUMBER_BETA}',
    f'- Class weights: {class_weight}',
    '- Oversampling: tidak digunakan',
    '- Focal loss: tidak digunakan',
    (
        '- Checkpoint robust: '
        f'{SELECTION_MACRO_WEIGHT:.0%} macro F1 + '
        f'{SELECTION_GEOMEAN_WEIGHT:.0%} geometric mean F1 + '
        f'{SELECTION_WORST_CLASS_WEIGHT:.0%} minimum class F1 + '
        f'{SELECTION_RECALL_LCB_WEIGHT:.0%} recall LCB90'
    ),
    f'- Early stopping aktif; fine-tuning maksimum {FINE_TUNE_EPOCHS} epoch',
    '',
    '## Final Test',
    f'- Accuracy: {accuracy:.4f}',
    f'- Macro precision: {macro_precision:.4f}',
    f'- Macro recall: {macro_recall:.4f}',
    f'- Macro F1: {macro_f1:.4f}',
    f"- Minimum class F1: {test_balanced_selection['min_class_f1']:.4f}",
    f"- Minimum class recall: {test_balanced_selection['min_class_recall']:.4f}",
    f"- Minimum recall LCB90: {test_balanced_selection['min_recall_lcb']:.4f}",
    f"- Geometric mean F1: {test_balanced_selection['geometric_mean_f1']:.4f}",
    f'- Hispa precision: {hispa_precision:.4f}',
    f'- Hispa recall: {hispa_recall:.4f}',
    f'- Hispa F1: {hispa_f1:.4f}',
    f'- Macro ROC-AUC OVR: {macro_roc_auc:.4f}',
    f'- Macro PR-AUC: {macro_pr_auc:.4f}',
    f'- ECE sesudah kalibrasi: {ece_after:.4f}',
    f'- Valid coverage: {accepted_coverage:.4f}',
    f'- Valid accuracy: {accepted_accuracy:.4f}',
    '',
    '## Batasan',
    '- Model hanya mengenali empat kelas dalam dataset RiceLeafs.',
    '- Confidence tinggi tidak menjamin prediksi benar.',
    '- Test tidak dipakai untuk selection pada notebook ini.',
    '- Revisi setelah melihat test memerlukan holdout eksternal baru untuk klaim unbiased.',
    '- Saran penanganan penyakit harus ditinjau ahli pertanian.',
]

model_card = '\n'.join(model_card_lines) + '\n'
(MODEL_DIR / 'MODEL_CARD.md').write_text(model_card, encoding='utf-8')
print(model_card)


## 30. ZIP Model Bundle

In [ ]:
# zip bundle
LOCAL_BUNDLE_DIR = Path('/content/riceleaf_model_bundle')
LOCAL_BUNDLE_ZIP = Path('/content/riceleaf_model_bundle.zip')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(MODEL_DIR, LOCAL_BUNDLE_DIR)
if LOCAL_BUNDLE_ZIP.exists():
    LOCAL_BUNDLE_ZIP.unlink()
with zipfile.ZipFile(LOCAL_BUNDLE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in LOCAL_BUNDLE_DIR.rglob('*'):
        if file_path.is_file():
            zip_file.write(file_path, arcname=file_path.relative_to(LOCAL_BUNDLE_DIR))
DRIVE_BUNDLE_ZIP = DRIVE_ROOT / 'riceleaf_model_bundle.zip'
shutil.copy2(LOCAL_BUNDLE_ZIP, DRIVE_BUNDLE_ZIP)
print('Model:', FINAL_MODEL_PATH)
print('Bundle:', DRIVE_BUNDLE_ZIP)
print('Evaluasi:', RESULT_DIR)

## 31. Fungsi Prediksi Colab

In [ ]:
# fungsi prediksi dengan TTA
def confidence_status(confidence, margin):
    if confidence < LOW_CONF_THRESHOLD:
        return 'rejected_low_confidence'
    if (
        confidence < HIGH_CONF_THRESHOLD
        or margin < MIN_CONFIDENCE_MARGIN
    ):
        return 'uncertain'
    return 'valid'

def prepare_uploaded_image(file_path):
    with Image.open(file_path) as image:
        original = image.convert('RGB')
        image_array = np.asarray(original)

    tensor = tf.image.resize_with_pad(
        image_array,
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        antialias=True,
    )
    tensor = tf.expand_dims(
        tf.cast(tensor, tf.float32),
        axis=0,
    )
    return original, tensor

def predict_uploaded_image(file_path):
    original, tensor = prepare_uploaded_image(
        file_path
    )

    variants = [tensor]
    if USE_TTA:
        variants.extend([
            tf.image.flip_left_right(tensor),
            tf.image.flip_up_down(tensor),
        ])

    variant_batch = tf.concat(
        variants,
        axis=0,
    )
    variant_logits = final_model.predict(
        variant_batch,
        verbose=0,
    )
    logits = np.mean(
        variant_logits,
        axis=0,
    )
    probabilities = softmax(
        logits / TEMPERATURE,
    )

    predicted_index = int(
        np.argmax(probabilities)
    )
    confidence = float(
        probabilities[predicted_index]
    )
    sorted_probabilities = np.sort(
        probabilities
    )
    margin = float(
        sorted_probabilities[-1]
        - sorted_probabilities[-2]
    )

    status = confidence_status(
        confidence,
        margin,
    )
    top_class = class_names[predicted_index]

    return original, {
        'status': status,
        'predicted_class': (
            None
            if status == 'rejected_low_confidence'
            else top_class
        ),
        'top_class_internal': top_class,
        'confidence': confidence,
        'margin': margin,
        'scores': {
            name: float(score)
            for name, score in zip(
                class_names,
                probabilities,
            )
        },
    }

## 32. Upload Gambar untuk Testing

In [ ]:
# upload dan testing gambar berulang
TEST_UPLOAD_DIR = Path('/content/test_uploads')
TEST_UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

test_history = []
test_round = 1

def show_uploaded_prediction(
    filename,
    image,
    result,
    round_number,
):
    print('\n' + '=' * 72)
    print(f'Putaran : {round_number}')
    print(f'File    : {filename}')
    print(
        json.dumps(
            result,
            indent=2,
            ensure_ascii=False,
        )
    )

    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.title(
        f"Status: {result['status']}\n"
        f"Hasil: {result['predicted_class']}\n"
        f"Top class: {result['top_class_internal']}\n"
        f"Confidence: {result['confidence']:.2%} | "
        f"Margin: {result['margin']:.2%}"
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    score_series = (
        pd.Series(result['scores'])
        .sort_values(ascending=False)
    )
    plt.figure(figsize=(8, 4))
    bars = plt.bar(
        score_series.index,
        score_series.values,
    )
    plt.ylim(0, 1)
    plt.ylabel('Probability')
    plt.title(
        'Confidence Seluruh Kelas '
        '(setelah TTA + kalibrasi)'
    )
    plt.xticks(rotation=20)

    for bar, value in zip(
        bars,
        score_series.values,
    ):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.02,
            f'{value:.1%}',
            ha='center',
            va='bottom',
        )

    plt.tight_layout()
    plt.show()

    return {
        'round': round_number,
        'filename': filename,
        'status': result['status'],
        'predicted_class': result['predicted_class'],
        'top_class': result['top_class_internal'],
        'confidence': result['confidence'],
        'margin': result['margin'],
    }

print('=' * 72)
print('MODE TESTING BERULANG AKTIF')
print('Upload satu atau beberapa gambar pada setiap putaran.')
print('Setelah hasil muncul, tekan Enter untuk upload lagi.')
print("Ketik 'q' untuk mengakhiri sesi testing.")
print('=' * 72)

while True:
    print(f'\nPUTARAN TEST {test_round}')
    uploaded_files = files.upload()

    if not uploaded_files:
        print('Tidak ada file yang dipilih.')
    else:
        for filename, content in uploaded_files.items():
            timestamp = datetime.now().strftime(
                '%Y%m%d_%H%M%S_%f'
            )
            safe_filename = Path(filename).name
            file_path = (
                TEST_UPLOAD_DIR
                / f'{timestamp}_{safe_filename}'
            )
            file_path.write_bytes(content)

            try:
                image, result = predict_uploaded_image(
                    file_path
                )
            except Exception as error:
                print(
                    safe_filename,
                    'gagal:',
                    error,
                )
                continue

            history_row = show_uploaded_prediction(
                safe_filename,
                image,
                result,
                test_round,
            )
            test_history.append(history_row)

    if test_history:
        print('\nRingkasan seluruh pengujian:')
        history_df = pd.DataFrame(test_history)
        display(
            history_df.style.format({
                'confidence': '{:.2%}',
                'margin': '{:.2%}',
            })
        )

    next_action = input(
        "\nTekan Enter untuk upload lagi, "
        "atau ketik 'q' untuk selesai: "
    ).strip().lower()

    if next_action in {
        'q',
        'quit',
        'selesai',
        'n',
        'no',
    }:
        break

    test_round += 1

print('\nSesi testing selesai.')
print(
    'Total gambar berhasil diuji:',
    len(test_history),
)

## 33. Ringkasan Akhir dan Keputusan Fit

Gunakan macro-F1, minimum class F1, recall per kelas, confusion matrix, serta gap validation-test untuk menentukan apakah model good fit. Accuracy saja tidak cukup.


In [ ]:
# ringkasan final
print('=' * 72)
print('TRAINING DAN EKSPOR V8 SELESAI')
print('Arsitektur  : ResNet50V2')
print('Input       :', IMAGE_SIZE)
print('Split seed  :', SPLIT_SEED)
print('Fingerprint :', split_fingerprint)
print('Model final :', FINAL_MODEL_PATH)
print('Bundle web  :', DRIVE_BUNDLE_ZIP)
print('Evaluasi    :', RESULT_DIR)
print('Data audit  :', AUDIT_DIR)
print('Accuracy    :', f"{accuracy:.4f}")
print('Macro F1    :', f"{macro_f1:.4f}")
print('Worst F1    :', f"{test_balanced_selection['min_class_f1']:.4f}")
print('Worst recall:', f"{test_balanced_selection['min_class_recall']:.4f}")
print('Recall LCB90:', f"{test_balanced_selection['min_recall_lcb']:.4f}")
print('Geomean F1  :', f"{test_balanced_selection['geometric_mean_f1']:.4f}")
print('Hispa F1    :', f"{hispa_f1:.4f}")
print('Temperature :', f"{TEMPERATURE:.4f}")
print('TTA selected:', USE_TTA)
print('Threshold   :', LOW_CONF_THRESHOLD, '/', HIGH_CONF_THRESHOLD)
print('Min margin  :', MIN_CONFIDENCE_MARGIN)
print('Valid       :', f"coverage={accepted_coverage:.4f},", f"accuracy={accepted_accuracy:.4f}")
print('Reviewable  :', f"coverage={reviewable_coverage:.4f},", f"accuracy={reviewable_accuracy:.4f}")
print('Test dipakai untuk selection: TIDAK')
print('=' * 72)
